In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Labs"
)

xpt_file = DATA_DIR / "FASTQX_D.xpt"
csv_file = DATA_DIR / "FASTQX_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for final fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 FASTQX_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if len(df) != 9440:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 19:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "FASTQX_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: FASTQX_D structure = "
    "9,440 rows x 19 columns."
)


# ============================================================
# 4. KNOWN PANDAS / XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. EXACT TINY COUNTS FOUND IN ATTACHED XPT
# ============================================================

EXPECTED_TINY_COUNTS = {

    "PHDSESN": 4409,

    "PHAFSTHR": 916,
    "PHAFSTMN": 164,

    "PHAGUMHR": 93,
    "PHAGUMMN": 2,

    "PHASUPHR": 4,
    "PHASUPMN": 1,

    "PHAALCHR": 2,

    "PHACOFHR": 1
}


ZERO_ALLOWED_COLUMNS = list(
    EXPECTED_TINY_COUNTS.keys()
)


# ============================================================
# 6. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


approved_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    approved_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "FASTQX_D fields:\n"
        + str(sorted(unexpected))
    )


missing_expected = (
    approved_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(tiny_counts.sum()) != 5592:

    raise ValueError(
        "Unexpected total number of tiny artifacts."
    )


print(
    "\nPASS: All 5,592 tiny values occur only "
    "in CDC zero-permitted fields."
)


# ============================================================
# 7. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )

    count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: {count:,} tiny -> 0"
    )


# ============================================================
# 8. VERIFY RESTORED ZEROS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


zero_failures = []


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: {actual_zero:,}"
    )


    if actual_zero != expected_count:

        zero_failures.append(
            (
                col,
                expected_count,
                actual_zero
            )
        )


if zero_failures:

    raise ValueError(
        "Zero restoration mismatch:\n"
        + str(zero_failures)
    )


print(
    "PASS: All 5,592 validated zeros restored."
)


# ============================================================
# 9. CONFIRM NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny values:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifacts remain."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 10. CHECK FOR REAL DECIMAL VALUES
# ============================================================

#
# FASTQX_D contains questionnaire codes,
# hours, minutes, and session codes.
#
# There should be NO true fractional values.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fraction_failures = []


for col in numeric_cols:

    values = df[col].dropna()


    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional.any():

        fraction_failures.append(
            col
        )

        print(
            f"\nUnexpected fractions in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )


if fraction_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in FASTQX_D:\n"
        + str(fraction_failures)
    )


print(
    "PASS: All FASTQX_D values are whole-number "
    "identifier/code/hour/minute values."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print("\n--- SEQN ---")


print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31128
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 12. CDC QUESTION FREQUENCY CHECKS
# ============================================================

EXPECTED_QUESTION_FREQUENCIES = {

    "PHQ020": {
        1: 26,
        2: 8877
    },

    "PHQ030": {
        1: 14,
        2: 8889
    },

    "PHQ040": {
        1: 146,
        2: 8757
    },

    "PHQ050": {
        1: 10,
        2: 8893
    },

    "PHQ060": {
        1: 28,
        2: 8875
    }
}


print("\n--- CDC QUESTION FREQUENCIES ---")


for col, expected in EXPECTED_QUESTION_FREQUENCIES.items():

    actual = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual != expected:

        raise ValueError(
            f"{col} frequency differs.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    if missing != 537:

        raise ValueError(
            f"{col}: expected 537 missing, "
            f"found {missing}."
        )


    print(
        f"PASS: {col} {actual}, missing=537"
    )


# ============================================================
# 13. CDC TIME/RANGE CHECKS
# ============================================================

TIME_CHECKS = {

    "PHACOFHR": {
        "min": 0,
        "max": 10,
        "valid": 26,
        "missing": 9414,
        "zero": 1
    },

    "PHACOFMN": {
        "min": 3,
        "max": 54,
        "valid": 26,
        "missing": 9414,
        "zero": 0
    },

    "PHAALCHR": {
        "min": 0,
        "max": 17,
        "valid": 14,
        "missing": 9426,
        "zero": 2
    },

    "PHAALCMN": {
        "min": 4,
        "max": 48,
        "valid": 14,
        "missing": 9426,
        "zero": 0
    },

    "PHAGUMHR": {
        "min": 0,
        "max": 13,
        "valid": 146,
        "missing": 9294,
        "zero": 93
    },

    "PHAGUMMN": {
        "min": 0,
        "max": 59,
        "valid": 146,
        "missing": 9294,
        "zero": 2
    },

    "PHAANTHR": {
        "min": 1,
        "max": 13,
        "valid": 10,
        "missing": 9430,
        "zero": 0
    },

    "PHAANTMN": {
        "min": 7,
        "max": 43,
        "valid": 10,
        "missing": 9430,
        "zero": 0
    },

    "PHASUPHR": {
        "min": 0,
        "max": 12,
        "valid": 28,
        "missing": 9412,
        "zero": 4
    },

    "PHASUPMN": {
        "min": 0,
        "max": 59,
        "valid": 28,
        "missing": 9412,
        "zero": 1
    },

    "PHAFSTHR": {
        "min": 0,
        "max": 86,
        "valid": 8903,
        "missing": 537,
        "zero": 916
    },

    "PHAFSTMN": {
        "min": 0,
        "max": 59,
        "valid": 8903,
        "missing": 537,
        "zero": 164
    }
}


print("\n--- CDC TIME/RANGE CHECKS ---")


for col, spec in TIME_CHECKS.items():

    s = df[col]


    if int(
        s.notna().sum()
    ) != spec["valid"]:

        raise ValueError(
            f"{col}: valid count differs."
        )


    if int(
        s.isna().sum()
    ) != spec["missing"]:

        raise ValueError(
            f"{col}: missing count differs."
        )


    if s.min() != spec["min"]:

        raise ValueError(
            f"{col}: minimum differs. "
            f"Expected {spec['min']}, got {s.min()}."
        )


    if s.max() != spec["max"]:

        raise ValueError(
            f"{col}: maximum differs. "
            f"Expected {spec['max']}, got {s.max()}."
        )


    if int(
        s.eq(0)
        .sum()
    ) != spec["zero"]:

        raise ValueError(
            f"{col}: zero count differs."
        )


    print(
        f"PASS: {col:<9}"
        f" range={s.min()}-{s.max()}"
        f" valid={int(s.notna().sum()):,}"
        f" missing={int(s.isna().sum()):,}"
        f" zeros={int(s.eq(0).sum()):,}"
    )


# ============================================================
# 14. CDC SESSION CHECK
# ============================================================

EXPECTED_PHDSESN = {
    0: 4409,
    1: 3326,
    2: 1705
}


actual_phdsesn = (
    df["PHDSESN"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_phdsesn != EXPECTED_PHDSESN:

    raise ValueError(
        "PHDSESN differs from CDC.\n"
        f"Expected: {EXPECTED_PHDSESN}\n"
        f"Actual:   {actual_phdsesn}"
    )


if int(
    df["PHDSESN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Unexpected missing PHDSESN."
    )


print(
    "\nPASS: PHDSESN matches CDC:"
)

print(
    "0 Morning   = 4,409"
)

print(
    "1 Afternoon = 3,326"
)

print(
    "2 Evening   = 1,705"
)


# ============================================================
# 15. BUILD EXPECTED FIDELITY SOURCE
# ============================================================

#
# Expected source =
#
# Original XPT
# +
# ONLY the 9 validated tiny -> 0 restorations
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 16. VERIFY VALUES BEFORE FORMATTING
# ============================================================

print("\n--- VALUE FIDELITY BEFORE FORMATTING ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected changes before formatting:\n"
        + str(value_failures)
    )


print(
    "PASS: All values equal original XPT "
    "after ONLY validated zero restoration."
)


# ============================================================
# 17. FORMAT SEQN + ALL QUESTIONNAIRE FIELDS
# ============================================================

#
# There are NO genuine decimal measurement fields
# in FASTQX_D.
#
# Therefore:
#
# 31128.0 -> 31128
# 1.0     -> 1
# 0.0     -> 0
# 59.0    -> 59
#
# No rounding is being done.
# We have already proven all nonmissing values
# are exact whole numbers.
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN/questionnaire/time fields "
    "formatted as whole numbers."
)


# ============================================================
# 18. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 19. RE-READ CSV FOR NUMERIC FIDELITY
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 19 CSV variables numerically "
    "match expected XPT values."
)


# ============================================================
# 21. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 22. RAW CSV TOKEN CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 23. EXACT RAW FORMAT CHECK
# ============================================================

#
# Every nonmissing FASTQX_D value should now be
# represented as an integer token.
#
# Examples:
#
# SEQN:     31128
# PHQ020:   2
# PHAFSTHR: 12
# PHDSESN:  0
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(raw_rows):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(int(source_value))
        )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(raw_examples) < 20:

                raw_examples.append(
                    (
                        i + 2,
                        col,
                        expected_text,
                        actual_text
                    )
                )


print(
    "\nRaw-format mismatches:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:",
        raw_examples
    )

    raise ValueError(
        "Raw CSV formatting differs."
    )


print(
    "PASS: All raw CSV values formatted correctly."
)


# ============================================================
# 24. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0


for row in raw_rows:

    for value in row.values():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 25. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for row in raw_rows:

    for value in row.values():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Scientific notation remains."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 26. FINAL TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


if "5.397605346934028e-79" in csv_text:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: Tiny XPORT artifact absent from CSV."
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FASTQX_D CONVERSION COMPLETE")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Validated tiny -> zero:", "5,592")
print("Affected variables:", 9)
print("Genuine decimal variables:", 0)
print("XPT -> CSV numeric differences: 0 expected")
print("Missing-position differences: 0 expected")
print("Raw-format differences: 0 expected")
print("Values ending .0: 0 expected")
print("Scientific notation: 0 expected")

NHANES 2005-2006 FASTQX_D XPT -> CSV
Rows: 9,440
Columns: 19
PASS: FASTQX_D structure = 9,440 rows x 19 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 5,592

Tiny values by variable:
PHDSESN     4409
PHAFSTHR     916
PHAFSTMN     164
PHAGUMHR      93
PHASUPHR       4
PHAGUMMN       2
PHAALCHR       2
PHACOFHR       1
PHASUPMN       1

PASS: All 5,592 tiny values occur only in CDC zero-permitted fields.

--- ZERO RESTORATION ---
PHDSESN   : 4,409 tiny -> 0
PHAFSTHR  : 916 tiny -> 0
PHAFSTMN  : 164 tiny -> 0
PHAGUMHR  : 93 tiny -> 0
PHAGUMMN  : 2 tiny -> 0
PHASUPHR  : 4 tiny -> 0
PHASUPMN  : 1 tiny -> 0
PHAALCHR  : 2 tiny -> 0
PHACOFHR  : 1 tiny -> 0

--- RESTORED ZERO COUNTS ---
PHDSESN   : 4,409
PHAFSTHR  : 916
PHAFSTMN  : 164
PHAGUMHR  : 93
PHAGUMMN  : 2
PHASUPHR  : 4
PHASUPMN  : 1
PHAALCHR  : 2
PHACOFHR  : 1
PASS: All 5,592 validated zeros restored.

Remaining tiny values: 0
PASS: No tiny-value artifacts remain.

--- FRACTIONAL VALUE CHECK 